# 02 · Method evaluation for the locked decisions (issue 008)

Issue: [Lock alignment, horizon buckets and band method](../issues/008-lock-alignment-horizon-and-band-method.md).
This notebook applies the decided methods **exactly as written in ADRs 0003, 0006, 0008 and 0009** and measures them.
It is evidence, not production code: the Metrics engine and Planning band tickets implement them in `backend/app/`.

Decisions applied (made by the project owner, 2026-10-25):
- **Alignment:** compare each hourly forecast with the actual half-hour that starts at the forecast timestamp (point).
- **Horizon buckets:** `[0,6) [6,12) [12,24) [24,36) [36,48) [48,72)` h.
- **Planning band:** global calibration, asymmetric empirical quantiles, 80 % nominal (90 % kept as the FR-3.3 toggle).
- **Decision view:** coverage = inside / valid hourly points; energy over valid hours only.
- **Bias:** reported separately from the band.

Run with `make notebook`. Inputs: the committed processed parquet files only.

In [1]:
import sys
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "backend"))
from app import config  # noqa: E402
from app.selection import parse_cutoff_clock, select_forecasts, target_grid  # noqa: E402

pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)

actuals = pd.read_parquet(config.PROCESSED_DIR / "actuals.parquet")
forecasts = pd.read_parquet(config.PROCESSED_DIR / "forecasts.parquet")
ACTUAL = actuals.set_index("start_time")["generation_mw"]
GRID = target_grid(config.DELIVERY_START, config.DELIVERY_END)
LEVELS = {80: (0.10, 0.90), 90: (0.05, 0.95)}  # = config.BAND_QUANTILES
MIN_CAL_N = 100  # proposed BAND_MIN_CALIBRATION_N (ADR 0006)
print(f"targets {len(GRID)}, forecast rows {len(forecasts)}, actual rows {len(actuals)}")

targets 744, forecast rows 15775, actual rows 1536


## 1 · Residuals (ADR 0003 + ADR 0006)

`e(t) = F(t) − A(t)`: F is the decision-time forecast from `app.selection` at the cutoff; A is the FUELHH half-hour
that **starts at t** (point alignment). A pair is *valid* only if both values exist; invalid pairs are counted and
excluded, never filled.

In [2]:
def residuals(cutoff: str) -> pd.DataFrame:
    out = select_forecasts(forecasts, GRID, parse_cutoff_clock(cutoff))
    out["actual_mw"] = out["start_time"].map(ACTUAL)
    out["error_mw"] = out["forecast_mw"] - out["actual_mw"]
    out["day"] = out["start_time"].dt.tz_convert(config.TZ).dt.date
    out["valid"] = out["forecast_mw"].notna() & out["actual_mw"].notna()
    return out


RES = {cut: residuals(cut) for cut in ("09:20", "09:50")}
pd.DataFrame(
    {
        cut: {
            "targets": len(r),
            "valid pairs": int(r["valid"].sum()),
            "missing forecast": int(r["forecast_mw"].isna().sum()),
            "missing actual": int(r["actual_mw"].isna().sum()),
        }
        for cut, r in RES.items()
    }
).T

,targets,valid pairs,missing forecast,missing actual
09:20,744,744,0,0
09:50,744,744,0,0


## 2 · Global calibration and the band (ADR 0006)

Quantiles are empirical with linear interpolation between order statistics (`numpy`/`pandas` default, Hyndman–Fan
type 7), over **all valid pairs of all in-scope delivery days at the selected cutoff**. The same quantiles apply to every
day, whatever date range the UI shows. Band: `lower = max(0, F − q_hi)`, `upper = max(0, F − q_lo)`. Coverage counts
`lower ≤ A ≤ upper` (inclusive) over valid points only.

In [3]:
def calibrate(errors: pd.Series, level: int) -> tuple[float, float] | None:
    """Return (q_lo, q_hi), or None when there are too few valid residuals."""
    e = errors.dropna()
    if len(e) < MIN_CAL_N:
        return None
    lo, hi = LEVELS[level]
    return float(e.quantile(lo)), float(e.quantile(hi))


def band(frame: pd.DataFrame, q: tuple[float, float]) -> pd.DataFrame:
    q_lo, q_hi = q
    lower = (frame["forecast_mw"] - q_hi).clip(lower=0)
    upper = (frame["forecast_mw"] - q_lo).clip(lower=0)
    inside = frame["actual_mw"].between(lower, upper) & frame["valid"]
    below = (frame["actual_mw"] < lower) & frame["valid"]
    return frame.assign(lower=lower, upper=upper, inside=inside, below=below)


rows = []
for cut, r in RES.items():
    for level in LEVELS:
        q = calibrate(r.loc[r["valid"], "error_mw"], level)
        b = band(r, q)
        n = int(b["valid"].sum())
        rows.append(
            {
                "cutoff": cut,
                "level": level,
                "q_lo": q[0],
                "q_hi": q[1],
                "n_cal": n,
                "width_mw": q[1] - q[0],
                "in-sample coverage": b["inside"].sum() / n,
                "actual below band": b["below"].sum() / n,
                "q_lo <= 0 <= q_hi": q[0] <= 0 <= q[1],
            }
        )
GLOBAL = pd.DataFrame(rows).set_index(["cutoff", "level"])
GLOBAL

q_lo     q_hi  n_cal  width_mw  in-sample coverage  actual below band  q_lo <= 0 <= q_hi
cutoff level                                                                                             
09:20  80     -426.40  4154.40    744    4580.8               0.798              0.101               True
       90    -1038.65  5268.35    744    6307.0               0.898              0.051               True
09:50  80     -502.70  4183.90    744    4686.6               0.798              0.101               True
       90    -1041.40  5319.00    744    6360.4               0.898              0.051               True

## 3 · Chronological held-out evaluation

Two checks; neither ever lets a delivery day be calibrated on its own residuals or on later days.

1. **Single split:** calibrate on delivery days 2–21 Jan (20 days), evaluate on 22 Jan–1 Feb (11 days).
2. **Rolling origin, decision-time honest:** for each delivery day D, calibrate on all delivery days **≤ D−2** (at the
   D−1 cutoff the actuals for D−1 are not yet complete) and evaluate on D. A day is evaluated only once it has at least
   7 calibration days, so D runs from 10 Jan to 1 Feb (23 days).

In [4]:
SPLIT = pd.Timestamp("2026-01-22").date()
rows = []
for cut, r in RES.items():
    train, test = r[r["day"] < SPLIT], r[r["day"] >= SPLIT]
    for level in LEVELS:
        q = calibrate(train.loc[train["valid"], "error_mw"], level)
        b = band(test, q)
        n = int(b["valid"].sum())
        rows.append(
            {
                "cutoff": cut,
                "level": level,
                "train days": train["day"].nunique(),
                "test days": test["day"].nunique(),
                "q_lo": q[0],
                "q_hi": q[1],
                "test n": n,
                "held-out coverage": b["inside"].sum() / n,
                "actual below band": b["below"].sum() / n,
            }
        )
SPLIT_EVAL = pd.DataFrame(rows).set_index(["cutoff", "level"])
SPLIT_EVAL

train days  test days     q_lo     q_hi  test n  held-out coverage  actual below band
cutoff level                                                                                       
09:20  80             20         11  -911.30  4160.20     264              0.902              0.098
       90             20         11 -1388.75  5080.15     264              0.928              0.072
09:50  80             20         11  -954.30  4191.60     264              0.902              0.098
       90             20         11 -1238.20  5179.45     264              0.939              0.061

In [5]:
def rolling(r: pd.DataFrame, level: int, min_days: int = 7) -> pd.DataFrame:
    days = sorted(r["day"].unique())
    out = []
    for d in days:
        history = r[(r["day"] <= d - timedelta(days=2)) & r["valid"]]
        if history["day"].nunique() < min_days:
            continue
        q = calibrate(history["error_mw"], level)
        b = band(r[r["day"] == d], q)
        out.append(
            {
                "day": d,
                "cal days": history["day"].nunique(),
                "q_lo": q[0],
                "q_hi": q[1],
                "valid": int(b["valid"].sum()),
                "inside": int(b["inside"].sum()),
                "below": int(b["below"].sum()),
            }
        )
    return pd.DataFrame(out).set_index("day")


rows, ROLL = [], {}
for cut, r in RES.items():
    for level in LEVELS:
        roll = ROLL[(cut, level)] = rolling(r, level)
        daily = roll["inside"] / roll["valid"]
        rows.append(
            {
                "cutoff": cut,
                "level": level,
                "eval days": len(roll),
                "pooled coverage": roll["inside"].sum() / roll["valid"].sum(),
                "actual below band": roll["below"].sum() / roll["valid"].sum(),
                "daily coverage min": daily.min(),
                "daily median": daily.median(),
                "days < 50 %": int((daily < 0.5).sum()),
                "calibrations with q_lo > 0": int((roll["q_lo"] > 0).sum()),
            }
        )
ROLL_EVAL = pd.DataFrame(rows).set_index(["cutoff", "level"])
ROLL_EVAL

eval days  pooled coverage  actual below band  daily coverage min  daily median  days < 50 %  calibrations with q_lo > 0
cutoff level                                                                                                                          
09:20  80            23            0.784              0.141               0.000           1.0            3                           0
       90            23            0.873              0.111               0.042           1.0            3                           0
09:50  80            23            0.768              0.149               0.000           1.0            3                           0
       90            23            0.879              0.111               0.083           1.0            3                           0

## 4 · PRD §11.2 test 5 against the band's meaning

PRD wording: *"Band: lower edge never below 0; band contains forecast."* The band is an interval for the **actual**
(`A = F − e`), built from signed-error quantiles. Before clipping it contains F exactly when `q_lo ≤ 0 ≤ q_hi`, which is
a property of the data, not of the method. A forecast that is too high more than 90 % of the time gives `q_lo > 0`,
and a correct band must then sit wholly below F.

In [6]:
global_holds = bool(GLOBAL["q_lo <= 0 <= q_hi"].all())
roll_920 = ROLL[("09:20", 80)]
excluded = roll_920[roll_920["q_lo"] > 0]
synthetic = band(
    pd.DataFrame({"forecast_mw": [10_000.0], "actual_mw": [8_000.0], "valid": [True]}),
    calibrate(pd.Series(np.linspace(500, 3_000, 200)), 80),
)
print(f"global calibrations (both cutoffs, 80/90): band contains F = {global_holds}")
print(
    f"rolling 09:20 80%: {len(excluded)} of {len(roll_920)} daily calibrations have q_lo > 0 "
    f"(band excludes F); q_lo range {roll_920['q_lo'].min():.0f}..{roll_920['q_lo'].max():.0f} MW"
)
synthetic["contains_F"] = synthetic["lower"].le(synthetic["forecast_mw"]) & synthetic["upper"].ge(
    synthetic["forecast_mw"]
)
cols = ["forecast_mw", "lower", "upper", "actual_mw", "inside", "contains_F"]
print("synthetic always-over-forecast case:", synthetic[cols].to_dict("records"))
r920 = RES["09:20"]
for days in (1, 2, 3):
    # Diagnostic only: raw q10 from short histories, ignoring the MIN_CAL_N guard (which needs
    # >= 100 residuals, about 5 days, and would refuse to draw these bands).
    q10s = []
    for d in sorted(r920["day"].unique()):
        hist = r920[(r920["day"] <= d - timedelta(days=2)) & r920["valid"]]
        if hist["day"].nunique() >= days:
            q10s.append(hist["error_mw"].quantile(0.10))
    print(
        f"short history >= {days} day(s): q10 > 0 (band would exclude F) in "
        f"{sum(q > 0 for q in q10s)} of {len(q10s)} calibrations"
    )

global calibrations (both cutoffs, 80/90): band contains F = True
rolling 09:20 80%: 0 of 23 daily calibrations have q_lo > 0 (band excludes F); q_lo range -1099..-282 MW
synthetic always-over-forecast case: [{'forecast_mw': 10000.0, 'lower': 7250.0, 'upper': 9250.0, 'actual_mw': 8000.0, 'inside': True, 'contains_F': False}]
short history >= 1 day(s): q10 > 0 (band would exclude F) in 3 of 29 calibrations
short history >= 2 day(s): q10 > 0 (band would exclude F) in 2 of 28 calibrations
short history >= 3 day(s): q10 > 0 (band would exclude F) in 1 of 27 calibrations


## 5 · Horizon buckets (ADR 0008)

All runs, cutoff-independent. The last bucket is `[48, 72)` and must contain every horizon above 48 h.

In [7]:
EDGES = [0, 6, 12, 24, 36, 48, 72]
runs = forecasts.assign(actual_mw=forecasts["start_time"].map(ACTUAL))
runs["error_mw"] = runs["generation_mw"] - runs["actual_mw"]
runs["uk_hour"] = runs["start_time"].dt.tz_convert(config.TZ).dt.hour
runs["bucket"] = pd.cut(runs["horizon_h"], EDGES, right=False)
by_bucket = runs.groupby("bucket", observed=False).agg(
    rows=("error_mw", "size"),
    valid=("error_mw", "count"),
    mae=("error_mw", lambda e: e.abs().mean()),
    bias=("error_mw", "mean"),
    uk_hours_present=("uk_hour", "nunique"),
)
print(
    f"horizons observed {runs['horizon_h'].min()}..{runs['horizon_h'].max()} h; "
    f"rows outside all buckets: {int(runs['bucket'].isna().sum())}; "
    f"bucket total {int(by_bucket['rows'].sum())} of {len(runs)}"
)
by_bucket

horizons observed 0.5..66.5 h; rows outside all buckets: 0; bucket total 15775 of 15775


,rows,valid,mae,bias,uk_hours_present
bucket,,,,,
"[0, 6)",1632,1494,1766.480,1547.883,24
"[6, 12)",1632,1504,1829.102,1613.788,24
"[12, 24)",3264,3050,1963.660,1788.338,24
"[24, 36)",3264,3072,2073.565,1797.368,24
"[36, 48)",3230,3040,2292.025,1968.722,24
"[48, 72)",2753,2591,2348.607,1484.151,19


## 6 · Decision-view KPIs (ADR 0009)

Per delivery day: **coverage** = inside / valid hourly points; **forecast energy** = Σ valid hourly forecast MW × 1 h;
**actual energy** = Σ hourly-average actual MW × 1 h over hours with both half-hours present. Energy comparisons use
only hours valid on both sides; a missing hour is never zero.

In [8]:
r = band(RES["09:20"], (GLOBAL.loc[("09:20", 80), "q_lo"], GLOBAL.loc[("09:20", 80), "q_hi"]))
r["actual_hour_mw"] = (
    r["start_time"].map(ACTUAL) + (r["start_time"] + pd.Timedelta(minutes=30)).map(ACTUAL)
) / 2
both = r["forecast_mw"].notna() & r["actual_hour_mw"].notna()
kpi = r.groupby("day").apply(
    lambda g: pd.Series(
        {
            "valid points": int(g["valid"].sum()),
            "inside": int(g["inside"].sum()),
            "energy hours": int(both[g.index].sum()),
            "forecast MWh": g.loc[both[g.index], "forecast_mw"].sum(),
            "actual MWh": g.loc[both[g.index], "actual_hour_mw"].sum(),
        }
    ),
    include_groups=False,
)
kpi["energy error %"] = (kpi["forecast MWh"] / kpi["actual MWh"] - 1) * 100
daily_cov = kpi["inside"] / kpi["valid points"]
print(
    f"days {len(kpi)}; days with < 24 valid points: {int((kpi['valid points'] < 24).sum())}; "
    f"days with < 24 energy hours: {int((kpi['energy hours'] < 24).sum())}"
)
print(
    f"daily coverage (inside / valid): median {daily_cov.median():.2f}, min {daily_cov.min():.2f}"
)
print(
    f"daily energy error: median {kpi['energy error %'].median():+.1f} %, "
    f"range {kpi['energy error %'].min():+.1f}..{kpi['energy error %'].max():+.1f} %"
)
kpi.head(5)

days 31; days with < 24 valid points: 0; days with < 24 energy hours: 0
daily coverage (inside / valid): median 0.92, min 0.04
daily energy error: median +13.9 %, range -11.2..+93.3 %


,valid points,inside,energy hours,forecast MWh,actual MWh,energy error %
day,,,,,,
2026-01-02,24.0,22.0,24.0,438696.0,364058.0,20.502
2026-01-03,24.0,21.0,24.0,416482.0,366029.0,13.784
2026-01-04,24.0,24.0,24.0,347753.0,320236.0,8.593
2026-01-05,24.0,16.0,24.0,225206.0,226546.5,-0.592
2026-01-06,24.0,20.0,24.0,270812.0,226764.5,19.424


## 7 · Bias, reported separately (ADR 0009)

Mean signed error over valid pairs. The data is **operationally metered wind** (about 82 % of GB wind, per Elexon);
the cause of the gap is unknown (assumptions Q-11).

In [9]:
pd.DataFrame(
    {
        cut: {
            "valid pairs": int(r["valid"].sum()),
            "mean forecast MW": r.loc[r["valid"], "forecast_mw"].mean(),
            "mean actual MW": r.loc[r["valid"], "actual_mw"].mean(),
            "bias MW": r.loc[r["valid"], "error_mw"].mean(),
            "bias % of mean actual": 100
            * r.loc[r["valid"], "error_mw"].mean()
            / r.loc[r["valid"], "actual_mw"].mean(),
        }
        for cut, r in RES.items()
    }
).T

,valid pairs,mean forecast MW,mean actual MW,bias MW,bias % of mean actual
09:20,744.0,13031.445,11285.507,1745.938,15.471
09:50,744.0,13013.489,11285.507,1727.983,15.312
